# Μάθημα 01 - Εισαγωγή στους Πράκτορες Τεχνητής Νοημοσύνης

Καλώς ήρθατε στο πρώτο μάθημα του μαθήματος **Πράκτορες Τεχνητής Νοημοσύνης για Αρχάριους**!

Ένας **πράκτορας ΤΝ** είναι ένα πρόγραμμα που χρησιμοποιεί ένα μεγάλο γλωσσικό μοντέλο (LLM) ως μηχανή συλλογισμού του και μπορεί να λαμβάνει *δράσεις* στον πραγματικό κόσμο — καλώντας APIs, ερωτώντας βάσεις δεδομένων ή τρέχοντας κώδικα — για να επιτύχει έναν στόχο εκ μέρους ενός χρήστη.

Σε αυτό το σημειωματάριο θα δημιουργήσετε τον πρώτο σας πράκτορα: έναν **Πράκτορα Ταξιδίων** που προτείνει προορισμούς διακοπών. Καθ’ οδόν θα μάθετε πώς να:

1. Συνδεθείτε στην υπηρεσία Azure AI Foundry Agent χρησιμοποιώντας το **Microsoft Agent Framework**.
2. Δώσετε στον πράκτορα ένα **εργαλείο** — μια απλή συνάρτηση Python που μπορεί να καλέσει.
3. Εκτελέσετε τον πράκτορα και να εξετάσετε την απάντησή του.
4. Μεταδώσετε την απάντηση του πράκτορα token-ανά-token.


## Ρύθμιση

Πριν εκτελέσετε αυτό το σημειωματάριο, βεβαιωθείτε ότι έχετε:

1. **Ένα έργο Azure AI Foundry** με ένα αναπτυγμένο μοντέλο συνομιλίας (π.χ. `gpt-4o-mini`).
2. **Συνδεθεί με το Azure CLI** — εκτελέστε `az login` στο τερματικό σας.
3. **Ορίσει τις απαραίτητες μεταβλητές περιβάλλοντος:**
   - `AZURE_AI_PROJECT_ENDPOINT` — το endpoint του έργου σας στο Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — το όνομα του αναπτυγμένου μοντέλου σας.

Το κελί παρακάτω εγκαθιστά τα πακέτα Python που χρειάζεστε.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Δημιουργία του Πρώτου Σας Πράκτορα

Ένας πράκτορας χρειάζεται δύο πράγματα:

- **Οδηγίες** που του λένε *ποιος είναι* και *πώς να συμπεριφέρεται* (μια οδηγία συστήματος).
- **Εργαλεία** — συναρτήσεις Python διακοσμημένες με `@tool` που ο πράκτορας μπορεί να καλέσει για να ανακτήσει πληροφορίες ή να εκτελέσει ενέργειες.

Παρακάτω ορίζουμε ένα απλό εργαλείο που επιστρέφει μια λίστα δημοφιλών προορισμών διακοπών. Ο πράκτορας θα χρησιμοποιήσει αυτό το εργαλείο όταν ένας χρήστης ζητήσει προτάσεις για ταξίδια.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Για μια πιο διαδραστική εμπειρία μπορείτε να **μεταδώσετε συνεχώς (stream)** την απάντηση του πράκτορα. Αντί να περιμένετε την πλήρη απάντηση, ο πράκτορας παρέχει τμήματα κειμένου καθώς παράγονται. Αυτό είναι ιδιαίτερα χρήσιμο σε διεπαφές συνομιλίας όπου θέλετε να εμφανίζετε την έξοδο σε πραγματικό χρόνο.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

Σε αυτό το μάθημα μάθατε πώς να:

- **Δημιουργήσετε έναν πάροχο** που συνδέεται με την υπηρεσία Azure AI Foundry Agent μέσω `AzureAIProjectAgentProvider`.
- **Ορίσετε ένα εργαλείο** χρησιμοποιώντας τον διακοσμητή `@tool` ώστε ο πράκτορας να μπορεί να καλεί τις συναρτήσεις Python σας.
- **Εκτελέσετε τον πράκτορα** με ένα μήνυμα χρήστη και να εκτυπώσετε την απάντησή του.
- **Ρεύμα απαντήσεων** για έξοδο σε πραγματικό χρόνο.

Στο επόμενο μάθημα θα εξερευνήσουμε τα πλαίσια agentic πιο αναλυτικά και θα μάθουμε πώς να δώσουμε στους πράκτορες πιο ισχυρά εργαλεία και δυνατότητες πολυβηματικής λογικής.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
